# 02 — Run Models

**Purpose:** send every question in `data/test_questions.csv` to a chatbot model and record its
response, for each model being compared.

[OpenRouter](https://openrouter.ai/) is used as the way to actually query the models. It exposes a
single OpenAI-compatible API in front of many different providers' models (Google Gemma, NVIDIA
Nemotron, OpenAI's open-weight gpt-oss, etc.), including free tiers for several of them. That means
one API key and one client setup can test several different chatbots without writing separate
integration code per provider — which matters here because the whole point of the project is
comparing *multiple* models against the same trustworthiness test set, not evaluating just one.

Before OpenRouter was adopted, the project's first sanity check queried Gemini directly — that
script is kept below as the earliest connectivity check, followed by the equivalent OpenRouter
check, then the actual full test run.


## 1. Quick connectivity check — Gemini direct

The very first "is my API key working" check, from before OpenRouter was introduced. Talks to
Google's Gemini API directly via the `google-genai` SDK.


In [ ]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()  # reads your .env file

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello in one sentence."
)

print(response.text)


## 2. Quick connectivity check — OpenRouter

The equivalent sanity check against OpenRouter, using the OpenAI SDK pointed at OpenRouter's
`base_url`. `openai/gpt-oss-20b:free` is used here just to confirm the key and endpoint work.


In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b:free",
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ]
)

print(response.choices[0].message.content)


## 3. Full test run — `run_tests_openrouter.py`

Runs every question in `data/test_questions.csv` against a chosen model and saves the responses to
`data/results_<model>.csv`. Two behaviours worth calling out:

- **Resumable**: if a results file already exists, only rows whose `response` starts with
  `"ERROR"` are retried — a completed run is never re-sent to the API.
- **Retries with backoff**: each question gets up to 3 attempts (5s apart) before being recorded as
  an `ERROR` row, since the free-tier OpenRouter models are prone to rate-limiting.

The original script took the model name as a command-line argument
(`python run_tests_openrouter.py google/gemma-4-26b-a4b-it:free`) so it could be re-run per model.
In notebook form that argument is set as a plain variable instead — change `model_name` and re-run
this cell for each model you want to evaluate (this project used it for Gemma, Nemotron, and
gpt-oss).


In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import pandas as pd
import time

load_dotenv()
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

# Set the OpenRouter model slug to test (originally passed as sys.argv[1] on the command line)
model_name = "google/gemma-4-26b-a4b-it:free"  # e.g. "google/gemma-4-31b:free"
safe_model_name = model_name.replace("/", "_").replace(":", "_")

results_path = f"data/results_{safe_model_name}.csv"
questions_path = "data/test_questions.csv"

questions_df = pd.read_csv(questions_path)

if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
    failed_mask = results_df["response"].str.startswith("ERROR", na=False)
    to_retry_ids = results_df.loc[failed_mask, "id"].tolist()
else:
    results_df = pd.DataFrame(columns=["id", "category", "question", "response"])
    to_retry_ids = questions_df["id"].tolist()

questions_to_run = questions_df[questions_df["id"].isin(to_retry_ids)]

for index, row in questions_to_run.iterrows():
    print(f"Testing question {row['id']}: {row['question'][:50]}...")

    answer = None
    last_error = None
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": row["question"]}]
            )
            answer = response.choices[0].message.content
            break
        except Exception as e:
            last_error = e
            print(f"  Attempt {attempt + 1} failed: {e}")
            time.sleep(5)

    if answer is None:
        answer = f"ERROR: Failed after 3 attempts. Last error: {last_error}"

    if row["id"] in results_df["id"].values:
        results_df.loc[results_df["id"] == row["id"], "response"] = answer
    else:
        new_row = {"id": row["id"], "category": row["category"], "question": row["question"], "response": answer}
        results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

    time.sleep(3)

results_df = results_df.sort_values("id")
results_df.to_csv(results_path, index=False)
print(f"\nDone! Results saved to {results_path}")
